# Modelo básico: riesgo difuso dentro del fitness de un AG
Versión mínima y autocontenida (sin descargar datos del OECE) para **ver cómo funciona** la idea:

`fitness(portafolio) = Σ compatibilidad − λ · riesgo_difuso(portafolio)`

- El **riesgo** (0-100) sale de un sistema difuso Mamdani con 4 entradas del portafolio: cantidad de licitaciones, capital comprometido, personal requerido y conflicto de fechas.
- Como es una **penalidad**, se resta: más riesgo → menor fitness.
- Con 8 licitaciones (2⁸ = 256 combinaciones) se puede comparar el AG contra la búsqueda exhaustiva.

## 1. Datos de ejemplo (inventados)

In [ ]:
import itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
rng = np.random.default_rng(42)

EMPRESA = {"capital": 1_000_000, "personal": 6, "k_max": 4}   # capital en S/, k_max = máx. de licitaciones a la vez

L = pd.DataFrame({
    "monto":    [600_000, 550_000, 300_000, 250_000, 150_000, 120_000, 400_000,  80_000],
    "score":    [     90,      85,      78,      74,      70,      66,      80,      55],  # compatibilidad (salida del 1.er sistema difuso)
    "personal": [      4,       4,       2,       2,       1,       1,       3,       1],
    "inicio":   [      0,      10,      20,      60,      70,      15,      30,      90],   # día de inicio de ejecución
    "duracion": [    120,     100,      90,      60,      60,      45,      80,      30],
}, index=[f"L{i+1}" for i in range(8)])
L["fin"] = L["inicio"] + L["duracion"]
N = len(L)

MONTO, S, PERS = L["monto"].values.astype(float), L["score"].values.astype(float), L["personal"].values.astype(float)

# conflicto de fechas entre pares = solape de ejecución / duración más corta  (0 a 1)
ini, fin = L["inicio"].values, L["fin"].values
inter = np.maximum(0, np.minimum(fin[:, None], fin[None, :]) - np.maximum(ini[:, None], ini[None, :]))
CONF = inter / np.minimum(L["duracion"].values[:, None], L["duracion"].values[None, :])
CONF_SUP = np.triu(CONF, 1)
L

## 2. Sistema difuso de riesgo (Mamdani)
Funciones de pertenencia triangulares, inferencia min-max y defuzzificación por centroide.

In [ ]:
def trimf(x, a, b, c):
    """Triangular con hombros: (a,a,c) es hombro izquierdo, (a,c,c) hombro derecho."""
    x = np.asarray(x, dtype=float)
    izq = (x - a) / (b - a) if b > a else np.ones_like(x)
    der = (c - x) / (c - b) if c > b else np.ones_like(x)
    return np.clip(np.minimum(izq, der), 0, 1)

# Entradas: qué tan "cargada" queda la empresa con el portafolio
VARS = {
    "cantidad": {"baja": (0, 0, 0.5),    "media": (0.3, 0.6, 0.85),  "alta": (0.6, 1, 1)},        # nº concursos / k_max
    "capital":  {"holgado": (0, 0, 0.6), "ajustado": (0.4, 0.8, 1.2), "excedido": (1, 2, 2)},     # monto total / capital
    "personal": {"holgado": (0, 0, 0.7), "justo": (0.5, 0.9, 1.3),   "excedido": (1, 2, 2)},      # personal req. / disponible
    "fechas":   {"bajo": (0, 0, 0.3),    "medio": (0.15, 0.4, 0.7),  "alto": (0.5, 1, 1)},        # conflicto promedio de pares
}
Y = np.linspace(0, 100, 201)
SALIDA = {"bajo": (0, 0, 30), "medio": (15, 40, 65), "alto": (50, 70, 90), "critico": (80, 100, 100)}
MF_SAL = {k: trimf(Y, *p) for k, p in SALIDA.items()}

REGLAS = [
    ({"capital": "excedido"}, "critico"),
    ({"personal": "excedido"}, "alto"),
    ({"cantidad": "alta", "fechas": "alto"}, "critico"),
    ({"capital": "ajustado", "cantidad": "alta"}, "alto"),
    ({"personal": "justo", "cantidad": "alta"}, "alto"),
    ({"capital": "ajustado", "personal": "justo"}, "alto"),
    ({"fechas": "alto"}, "alto"),
    ({"capital": "ajustado", "cantidad": "media"}, "medio"),
    ({"personal": "justo", "cantidad": "media"}, "medio"),
    ({"fechas": "medio", "cantidad": "alta"}, "medio"),
    ({"cantidad": "alta", "capital": "holgado", "personal": "holgado"}, "medio"),   # más concursos = más riesgo aunque sobren recursos
    ({"capital": "holgado", "personal": "holgado", "fechas": "bajo", "cantidad": "media"}, "bajo"),
    ({"cantidad": "baja"}, "bajo"),
    ({"capital": "holgado", "cantidad": "media"}, "bajo"),
    ({"personal": "holgado", "fechas": "bajo"}, "bajo"),
]

def entradas(x):
    k = float(x.sum()); pares = k * (k - 1) / 2
    return {
        "cantidad": min(1.0, k / EMPRESA["k_max"]),
        "capital":  min(2.0, float(MONTO @ x) / EMPRESA["capital"]),
        "personal": min(2.0, float(PERS @ x) / EMPRESA["personal"]),
        "fechas":   min(1.0, float(x @ CONF_SUP @ x) / pares) if pares > 0 else 0.0,
    }

def riesgo_difuso(x):
    if x.sum() == 0:
        return 0.0
    ent = entradas(x)
    grados = {v: {t: float(trimf(ent[v], *p)) for t, p in terms.items()} for v, terms in VARS.items()}
    agregada = np.zeros_like(Y)
    for ant, cons in REGLAS:
        w = min(grados[v][t] for v, t in ant.items())          # AND = mínimo
        if w > 0:
            agregada = np.maximum(agregada, np.minimum(w, MF_SAL[cons]))  # implicación min, agregación max
    return float((Y * agregada).sum() / agregada.sum()) if agregada.sum() > 0 else 0.0

# Prueba rápida: el riesgo crece con la cantidad de licitaciones
for k in range(1, 5):
    x = np.zeros(N, dtype=int); x[np.argsort(-S)[:k]] = 1
    print(f"Top-{k} por compatibilidad -> riesgo = {riesgo_difuso(x):5.1f}  | entradas = { {a: round(b, 2) for a, b in entradas(x).items()} }")

## 3. Algoritmo genético con riesgo en el fitness
Cromosoma binario de longitud 8 (1 = postular). Restricción dura: máximo `k_max` licitaciones (se repara quitando la de menor compatibilidad). `λ = 0` reproduce el AG **sin** riesgo, para comparar.

In [ ]:
def fitness(x, lam):
    return float(S @ x) - lam * riesgo_difuso(x)

def reparar(x):
    x = x.copy()
    while x.sum() > EMPRESA["k_max"]:
        act = np.flatnonzero(x)
        x[act[np.argmin(S[act])]] = 0
    if x.sum() == 0:
        x[np.argmax(S)] = 1
    return x

def ag(lam, pob_n=40, gens=60, p_cruce=0.9, p_mut=1/8, elite=2, seed=42):
    rng = np.random.default_rng(seed)
    pob = np.array([reparar((rng.random(N) < 0.4).astype(int)) for _ in range(pob_n)])
    hist = []
    for _ in range(gens):
        fit = np.array([fitness(x, lam) for x in pob])
        hist.append(fit.max())
        nueva = [pob[i].copy() for i in np.argsort(-fit)[:elite]]
        while len(nueva) < pob_n:
            padres = []
            for _ in range(2):                                    # torneo de 3
                idx = rng.choice(pob_n, 3, replace=False)
                padres.append(pob[idx[np.argmax(fit[idx])]])
            if rng.random() < p_cruce:                            # cruce uniforme
                m = rng.random(N) < 0.5
                hijos = [np.where(m, *padres), np.where(m, padres[1], padres[0])]
            else:
                hijos = [padres[0].copy(), padres[1].copy()]
            for h in hijos:
                flip = rng.random(N) < p_mut                      # mutación bit a bit
                h[flip] = 1 - h[flip]
                nueva.append(reparar(h))
        pob = np.array(nueva[:pob_n])
    fit = np.array([fitness(x, lam) for x in pob])
    return pob[np.argmax(fit)], hist

def exhaustivo(lam):
    mejor, xb = -np.inf, None
    for bits in itertools.product([0, 1], repeat=N):
        x = np.array(bits)
        if x.sum() == 0 or x.sum() > EMPRESA["k_max"]:
            continue
        f = fitness(x, lam)
        if f > mejor:
            mejor, xb = f, x
    return xb, mejor

## 4. Comparación: sin riesgo vs. con riesgo

In [ ]:
filas = []
for lam in [0.0, 1.0, 2.0, 3.0, 5.0]:
    x, hist = ag(lam)
    xb, fb = exhaustivo(lam)
    filas.append({
        "λ riesgo": lam,
        "portafolio (AG)": " + ".join(L.index[x == 1]),
        "Σ compat.": round(float(S @ x), 1),
        "riesgo": round(riesgo_difuso(x), 1),
        "fitness": round(fitness(x, lam), 1),
        "¿AG = óptimo?": abs(fitness(x, lam) - fb) < 1e-6,
    })
pd.DataFrame(filas).set_index("λ riesgo")

In [ ]:
# Detalle del portafolio elegido con λ = 2
x, hist = ag(2.0)
print("Portafolio:", list(L.index[x == 1]))
print("Entradas al sistema de riesgo:", {a: round(b, 2) for a, b in entradas(x).items()})
print(f"Σ compatibilidad = {S @ x:.1f} | riesgo = {riesgo_difuso(x):.1f} | fitness = {fitness(x, 2.0):.1f}")

plt.figure(figsize=(6, 3.5))
plt.plot(hist); plt.xlabel("Generación"); plt.ylabel("Mejor fitness"); plt.title("Convergencia (λ = 2)")
plt.grid(alpha=0.3); plt.show()

## 5. Cómo leerlo y qué ajustar
- Con `λ = 0` el AG elige las licitaciones de mayor compatibilidad aunque la empresa quede sobrecargada. Al subir `λ` el riesgo empieza a pesar y el AG cambia a portafolios más livianos; el punto de cambio depende de la escala de tus puntajes (aquí ocurre alrededor de λ = 2). Si con tus datos nunca cambia, sube `λ`; si siempre elige 1 solo concurso, bájalo.
- **Ajustables**: `λ` (peso del riesgo), las funciones de pertenencia en `VARS` / `SALIDA`, y las `REGLAS`.
- **Ideas para ampliar**: agregar entradas al riesgo (p. ej. dependencia de una sola entidad compradora, garantías requeridas) o conectar este mismo bloque al notebook completo (ya está integrado en `Portafolio_Concursos_Difusa_AG_Riesgo.ipynb`).
- Las reglas y umbrales son supuestos razonables, no calibrados con datos reales.